# Silver → Gold | State of Data Brasil

Notebook para execução no **AWS Glue Notebook**, utilizando **PySpark**.

## Objetivo

Ler a tabela Silver consolidada e gerar tabelas Gold analíticas, prontas para consultas no Athena,
gráficos e construção do material executivo do Tech Challenge.

### Entrada

```text
workspace.tb_state_of_data_silver
```

### Saídas Gold

```text
gold/
├── mercado/
├── diversidade/
├── tecnologias/
├── ia/
├── segmentacoes/
└── desafios/
```

### Tabelas no Glue Data Catalog

```text
workspace.gold_mercado
workspace.gold_diversidade
workspace.gold_tecnologias
workspace.gold_ia
workspace.gold_segmentacoes
workspace.gold_desafios
```

A Gold deixa de trabalhar no nível individual do respondente e passa a disponibilizar
**indicadores agregados** para responder às perguntas de negócio.

## Estrutura do notebook

1. Inicialização do AWS Glue  
2. Parâmetros do projeto  
3. Leitura da Silver  
4. Validação simples da entrada  
5. Funções auxiliares  
6. Gold Mercado  
7. Gold Diversidade  
8. Gold Tecnologias  
9. Gold Inteligência Artificial  
10. Gold Segmentações  
11. Gold Desafios  
12. Controle de qualidade  
13. Escrita das tabelas Gold no S3  
14. Catalogação no Glue Data Catalog  
15. Validação final  
16. Consultas sugeridas para Athena

## 1. Inicialização do AWS Glue

In [1]:
from awsglue.context import GlueContext
from pyspark.context import SparkContext
from pyspark.sql import functions as F
from pyspark.sql import types as T

import boto3

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

print("Spark:", spark.version)
print("Sessão pronta.")

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: dec1bbb7-7b78-4123-9985-452e8bf7ae4f
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session dec1bbb7-7b78-4123-9985-452e8bf7ae4f to get into ready status...
Session dec1bbb7-7b78-4123-9985-452e8bf7ae4f has been created.
Spark: 3.3.0-amzn-1
Sessão pronta.


## 2. Parâmetros do projeto

In [ ]:
DATABASE = "workspace"

SILVER_TABLE = "tb_state_of_data_silver"

BUCKET = "tech-challenge-fase3-015006598133"

GOLD_CONFIG = {
    "tb_gold_mercado": {
        "path": f"s3://{BUCKET}/gold/mercado/"
    },

    "tb_gold_diversidade": {
        "path": f"s3://{BUCKET}/gold/diversidade/"
    },

    "tb_gold_tecnologias": {
        "path": f"s3://{BUCKET}/gold/tecnologias/"
    },

    "tb_gold_ia": {
        "path": f"s3://{BUCKET}/gold/ia/"
    },

    "tb_gold_segmentacoes": {
        "path": f"s3://{BUCKET}/gold/segmentacoes/"
    },

    "tb_gold_desafios": {
        "path": f"s3://{BUCKET}/gold/desafios/"
    },
}

for tabela, config in GOLD_CONFIG.items():
    print(tabela, "->", config["path"])

tb_gold_mercado -> s3://tech-challenge-fase3-015006598133/gold/mercado/
tb_gold_diversidade -> s3://tech-challenge-fase3-015006598133/gold/diversidade/
tb_gold_tecnologias -> s3://tech-challenge-fase3-015006598133/gold/tecnologias/
tb_gold_ia -> s3://tech-challenge-fase3-015006598133/gold/ia/
tb_gold_segmentacoes -> s3://tech-challenge-fase3-015006598133/gold/segmentacoes/
tb_gold_desafios -> s3://tech-challenge-fase3-015006598133/gold/desafios/


## 3. Leitura da camada Silver pelo Glue Data Catalog

In [ ]:
df_silver = glueContext.create_data_frame_from_catalog(
    database=DATABASE,
    table_name=SILVER_TABLE
).cache()

print("Registros Silver:", df_silver.count())
print("Colunas Silver:", len(df_silver.columns))
df_silver.printSchema()

Registros Silver: 14002
Colunas Silver: 29
root
 |-- respondent_id: string (nullable = true)
 |-- faixa_idade: string (nullable = true)
 |-- genero: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- nivel_ensino: string (nullable = true)
 |-- area_formacao: string (nullable = true)
 |-- situacao_trabalho: string (nullable = true)
 |-- setor: string (nullable = true)
 |-- cargo_atual: string (nullable = true)
 |-- senioridade: string (nullable = true)
 |-- faixa_salarial: string (nullable = true)
 |-- tempo_experiencia_dados: string (nullable = true)
 |-- modelo_trabalho: string (nullable = true)
 |-- desafios_gestor: string (nullable = true)
 |-- ia_prioridade_empresa: string (nullable = true)
 |-- ia_resultados_empresa: string (nullable = true)
 |-- ia_motivos_nao_uso_empresa: string (nullable = true)
 |-- linguagens_uso: string (nullable = true)
 |-- bancos_dados_uso: string (nullable = true)
 |-- cloud_uso: string (nullable = true)

## 4. Validação simples da entrada

Nesta etapa apenas confirmamos:

- presença dos três anos;
- quantidade de registros;
- existência das principais variáveis utilizadas na Gold.

In [ ]:
df_silver.groupBy("ano").count().orderBy("ano").show()


CAMPOS_PRINCIPAIS = [
    "genero",
    "regiao",
    "setor",
    "grupo_cargo",
    "senioridade_grupo",
    "modelo_trabalho_grupo",
    "salario_estimado",
    "linguagens_uso",
    "bancos_dados_uso",
    "cloud_uso",
    "ferramentas_bi_uso",
    "ia_adocao",
    "ia_prioridade_empresa",
    "ia_resultados_empresa",
    "ia_motivos_nao_uso_empresa",
    "desafios_gestor",
]

print("\nCampos utilizados na Gold:")

for campo in CAMPOS_PRINCIPAIS:
    print(f"{campo:35s}", "OK" if campo in df_silver.columns else "NÃO ENCONTRADO")

+----+-----+
| ano|count|
+----+-----+
|2023| 5293|
|2024| 5215|
|2025| 3494|
+----+-----+


Campos utilizados na Gold:
genero                              OK
regiao                              OK
setor                               OK
grupo_cargo                         OK
senioridade_grupo                   OK
modelo_trabalho_grupo               OK
salario_estimado                    OK
linguagens_uso                      OK
bancos_dados_uso                    OK
cloud_uso                           OK
ferramentas_bi_uso                  OK
ia_adocao                           OK
ia_prioridade_empresa               OK
ia_resultados_empresa               OK
ia_motivos_nao_uso_empresa          OK
desafios_gestor                     OK


## 5. Funções auxiliares

As funções abaixo são utilizadas apenas para evitar repetição de código.

Elas calculam:

- quantidade de profissionais;
- participação percentual;
- salário médio;
- salário mediano;
- adoção de IA;
- separação de respostas multisseleção.

In [ ]:
def resumo_dimensao(df, coluna, dimensao):

    base = df.filter(
        F.col(coluna).isNotNull()
    )

    totais = (
        base
        .groupBy("ano")
        .agg(
            F.count("*").alias(
                "total_respondentes"
            )
        )
    )

    resumo = (
        base
        .groupBy(
            "ano",
            F.col(coluna).alias("categoria")
        )
        .agg(
            F.count("*").alias(
                "quantidade_profissionais"
            ),
            F.round(
                F.avg("salario_estimado"),
                2
            ).alias(
                "salario_medio"
            ),
            F.round(
                F.expr(
                    "percentile_approx("
                    "salario_estimado, 0.5)"
                ),
                2
            ).alias(
                "salario_mediano"
            )
        )
        .join(
            totais,
            on="ano",
            how="left"
        )
        .withColumn(
            "percentual",
            F.round(
                F.col(
                    "quantidade_profissionais"
                )
                / F.col(
                    "total_respondentes"
                )
                * 100,
                2
            )
        )
        .withColumn(
            "dimensao",
            F.lit(dimensao)
        )
        .select(
            "ano",
            "dimensao",
            "categoria",
            "quantidade_profissionais",
            "percentual",
            "salario_medio",
            "salario_mediano",
        )
    )

    return resumo


def resumo_segmentacao(df, coluna, dimensao):
    """
    Resume região, senioridade ou modelo de trabalho
    incluindo remuneração e índice de adoção de IA.
    """

    base = df.filter(
        F.col(coluna).isNotNull()
    )

    totais = (
        base
        .groupBy("ano")
        .agg(
            F.count("*").alias(
                "total_respondentes"
            )
        )
    )

    resumo = (
        base
        .groupBy(
            "ano",
            F.col(coluna).alias("categoria")
        )
        .agg(
            F.count("*").alias(
                "quantidade_profissionais"
            ),
            F.round(
                F.avg("salario_estimado"),
                2
            ).alias(
                "salario_medio"
            ),
            F.round(
                F.expr(
                    "percentile_approx("
                    "salario_estimado, 0.5)"
                ),
                2
            ).alias(
                "salario_mediano"
            ),
            F.round(
                F.avg(
                    F.when(
                        F.col("ia_adocao") == "Utiliza",
                        1.0
                    )
                    .when(
                        F.col("ia_adocao") == "Não utiliza",
                        0.0
                    )
                ) * 100,
                2
            ).alias(
                "percentual_adocao_ia"
            )
        )
        .join(
            totais,
            on="ano",
            how="left"
        )
        .withColumn(
            "percentual",
            F.round(
                F.col(
                    "quantidade_profissionais"
                )
                / F.col(
                    "total_respondentes"
                )
                * 100,
                2
            )
        )
        .withColumn(
            "dimensao",
            F.lit(dimensao)
        )
        .select(
            "ano",
            "dimensao",
            "categoria",
            "quantidade_profissionais",
            "percentual",
            "salario_medio",
            "salario_mediano",
            "percentual_adocao_ia",
        )
    )

    return resumo


def explodir_multiselect(df, coluna, nome_saida):
    """
    Converte uma resposta multisseleção em várias linhas.

    Exemplo:
    'SQL, Python, R'
        ->
    SQL
    Python
    R
    """

    return (
        df
        .filter(
            F.col(coluna).isNotNull()
        )
        .select(
            "ano",
            "respondent_id",
            F.explode(
                F.split(
                    F.regexp_replace(
                        F.col(coluna),
                        r"""[\[\]'"]""",
                        ""
                    ),
                    r"\s*[,;]\s*"
                )
            ).alias(
                nome_saida
            )
        )
        .withColumn(
            nome_saida,
            F.trim(
                F.col(nome_saida)
            )
        )
        .filter(
            F.col(nome_saida) != ""
        )
    )


def resumo_indicador(df, coluna, indicador):
    """
    Resume uma variável categórica por ano.
    Utilizado principalmente na Gold de IA.
    """

    base = df.filter(
        F.col(coluna).isNotNull()
    )

    totais = (
        base
        .groupBy("ano")
        .agg(
            F.count("*").alias(
                "total_respostas"
            )
        )
    )

    return (
        base
        .groupBy(
            "ano",
            F.col(coluna).alias("categoria")
        )
        .agg(
            F.count("*").alias(
                "quantidade"
            )
        )
        .join(
            totais,
            on="ano",
            how="left"
        )
        .withColumn(
            "percentual",
            F.round(
                F.col("quantidade")
                / F.col("total_respostas")
                * 100,
                2
            )
        )
        .withColumn(
            "indicador",
            F.lit(indicador)
        )
        .select(
            "ano",
            "indicador",
            "categoria",
            "quantidade",
            "percentual",
        )
    )

## 6. Gold Mercado

### Objetivo

Responder principalmente:

- Como está estruturado o mercado brasileiro de Dados?
- Quais perfis profissionais são mais valorizados?

Serão analisadas as dimensões:

- cargo;
- senioridade;
- setor;
- tempo de experiência.

Para cada categoria serão calculados:

- quantidade de profissionais;
- participação percentual;
- salário médio;
- salário mediano.

In [ ]:
mercado_cargo = resumo_dimensao(df_silver, "grupo_cargo", "Cargo")
mercado_senioridade = resumo_dimensao(df_silver, "senioridade_grupo", "Senioridade")
mercado_setor = resumo_dimensao(df_silver, "setor", "Setor")
mercado_experiencia = resumo_dimensao(df_silver, "tempo_experiencia_dados", "Experiência em Dados")


gold_mercado = (
    mercado_cargo
    .unionByName(
        mercado_senioridade
    )
    .unionByName(
        mercado_setor
    )
    .unionByName(
        mercado_experiencia
    )
    .cache()
)

gold_mercado.orderBy("ano", "dimensao", F.desc("quantidade_profissionais")).show(30, truncate=False)

+----+--------------------+--------------------------------------+------------------------+----------+-------------+---------------+
|ano |dimensao            |categoria                             |quantidade_profissionais|percentual|salario_medio|salario_mediano|
+----+--------------------+--------------------------------------+------------------------+----------+-------------+---------------+
|2023|Cargo               |Análise de Dados                      |907                     |23.52     |7137.77      |7000.5         |
|2023|Cargo               |Outros                                |859                     |22.27     |7653.58      |5000.5         |
|2023|Cargo               |Ciência de Dados                      |687                     |17.81     |10532.53     |10000.5        |
|2023|Cargo               |Engenharia de Dados                   |684                     |17.73     |11586.1      |10000.5        |
|2023|Cargo               |Business Intelligence                 |506

## 7. Gold Diversidade

### Objetivo

Responder:

- Qual é o cenário de diversidade de gênero nas carreiras de dados?

A tabela será formada em três níveis:

- distribuição geral por gênero;
- gênero por cargo;
- gênero por senioridade.

Também serão calculados salário médio e mediano.

In [ ]:
# ---------------------------
# Distribuição geral
# ---------------------------

div_geral_base = (
    df_silver
    .filter(
        F.col("genero").isNotNull()
    )
)

div_geral_total = (
    div_geral_base
    .groupBy("ano")
    .agg(
        F.count("*").alias("total")
    )
)

div_geral = (
    div_geral_base
    .groupBy(
        "ano",
        "genero"
    )
    .agg(
        F.count("*").alias("quantidade"),
        F.round(
            F.avg("salario_estimado"),
            2
        ).alias("salario_medio"),
        F.round(
            F.expr(
                "percentile_approx("
                "salario_estimado, 0.5)"
            ),
            2
        ).alias("salario_mediano")
    )
    .join(
        div_geral_total,
        on="ano",
        how="left"
    )
    .withColumn(
        "percentual",
        F.round(
            F.col("quantidade")
            / F.col("total")
            * 100,
            2
        )
    )
    .withColumn(
        "contexto",
        F.lit("Geral")
    )
    .withColumn(
        "categoria_contexto",
        F.lit("Todos")
    )
    .select(
        "ano",
        "contexto",
        "categoria_contexto",
        "genero",
        "quantidade",
        "percentual",
        "salario_medio",
        "salario_mediano",
    )
)


# ---------------------------
# Gênero por cargo
# ---------------------------

div_cargo_base = (
    df_silver
    .filter(
        F.col("genero").isNotNull()
        & F.col("grupo_cargo").isNotNull()
    )
)

div_cargo_total = (
    div_cargo_base
    .groupBy(
        "ano",
        "grupo_cargo"
    )
    .agg(
        F.count("*").alias("total")
    )
)

div_cargo = (
    div_cargo_base
    .groupBy(
        "ano",
        "grupo_cargo",
        "genero"
    )
    .agg(
        F.count("*").alias("quantidade"),
        F.round(
            F.avg("salario_estimado"),
            2
        ).alias("salario_medio"),
        F.round(
            F.expr(
                "percentile_approx("
                "salario_estimado, 0.5)"
            ),
            2
        ).alias("salario_mediano")
    )
    .join(
        div_cargo_total,
        on=[
            "ano",
            "grupo_cargo"
        ],
        how="left"
    )
    .withColumn(
        "percentual",
        F.round(
            F.col("quantidade")
            / F.col("total")
            * 100,
            2
        )
    )
    .withColumn(
        "contexto",
        F.lit("Cargo")
    )
    .withColumnRenamed(
        "grupo_cargo",
        "categoria_contexto"
    )
    .select(
        "ano",
        "contexto",
        "categoria_contexto",
        "genero",
        "quantidade",
        "percentual",
        "salario_medio",
        "salario_mediano",
    )
)


# ---------------------------
# Gênero por senioridade
# ---------------------------

div_sen_base = (
    df_silver
    .filter(
        F.col("genero").isNotNull()
        & F.col("senioridade_grupo").isNotNull()
    )
)

div_sen_total = (
    div_sen_base
    .groupBy(
        "ano",
        "senioridade_grupo"
    )
    .agg(
        F.count("*").alias("total")
    )
)

div_senioridade = (
    div_sen_base
    .groupBy(
        "ano",
        "senioridade_grupo",
        "genero"
    )
    .agg(
        F.count("*").alias("quantidade"),
        F.round(
            F.avg("salario_estimado"),
            2
        ).alias("salario_medio"),
        F.round(
            F.expr(
                "percentile_approx("
                "salario_estimado, 0.5)"
            ),
            2
        ).alias("salario_mediano")
    )
    .join(
        div_sen_total,
        on=[
            "ano",
            "senioridade_grupo"
        ],
        how="left"
    )
    .withColumn(
        "percentual",
        F.round(
            F.col("quantidade")
            / F.col("total")
            * 100,
            2
        )
    )
    .withColumn(
        "contexto",
        F.lit("Senioridade")
    )
    .withColumnRenamed(
        "senioridade_grupo",
        "categoria_contexto"
    )
    .select(
        "ano",
        "contexto",
        "categoria_contexto",
        "genero",
        "quantidade",
        "percentual",
        "salario_medio",
        "salario_mediano",
    )
)


gold_diversidade = (
    div_geral
    .unionByName(div_cargo)
    .unionByName(div_senioridade)
    .cache()
)

gold_diversidade.orderBy("ano", "contexto", "categoria_contexto", F.desc("quantidade")).show(30, truncate=False)

+----+-----------+---------------------+--------------------+----------+----------+-------------+---------------+
|ano |contexto   |categoria_contexto   |genero              |quantidade|percentual|salario_medio|salario_mediano|
+----+-----------+---------------------+--------------------+----------+----------+-------------+---------------+
|2023|Cargo      |Analytics Engineering|Masculino           |105       |75.54     |11443.36     |10000.5        |
|2023|Cargo      |Analytics Engineering|Feminino            |33        |23.74     |9939.89      |7000.5         |
|2023|Cargo      |Analytics Engineering|Outro               |1         |0.72      |5000.5       |5000.5         |
|2023|Cargo      |Análise de Dados     |Masculino           |645       |71.11     |7222.21      |5000.5         |
|2023|Cargo      |Análise de Dados     |Feminino            |259       |28.56     |6917.49      |7000.5         |
|2023|Cargo      |Análise de Dados     |Outro               |2         |0.22      |8500.

## 8. Gold Tecnologias

### Objetivo

Responder:

- Quais tecnologias apresentam maior adoção entre os profissionais?

Serão consideradas:

- linguagens;
- bancos de dados;
- cloud;
- ferramentas de BI.

As respostas multisseleção são transformadas em linhas individuais e depois agregadas.

In [ ]:
def criar_gold_tecnologia(df, coluna, categoria_tecnologia):
    base_respondentes = (
        df
        .filter(
            F.col(coluna).isNotNull()
        )
        .groupBy("ano")
        .agg(
            F.count("*").alias(
                "respondentes_com_resposta"
            )
        )
    )

    explodido = explodir_multiselect(df, coluna, "tecnologia")

    resumo = (
        explodido
        .groupBy(
            "ano",
            "tecnologia"
        )
        .agg(
            F.count("*").alias(
                "quantidade_profissionais"
            )
        )
        .join(
            base_respondentes,
            on="ano",
            how="left"
        )
        .withColumn(
            "percentual_adocao",
            F.round(
                F.col(
                    "quantidade_profissionais"
                )
                / F.col(
                    "respondentes_com_resposta"
                )
                * 100,
                2
            )
        )
        .withColumn(
            "categoria_tecnologia",
            F.lit(
                categoria_tecnologia
            )
        )
        .select(
            "ano",
            "categoria_tecnologia",
            "tecnologia",
            "quantidade_profissionais",
            "percentual_adocao",
        )
    )

    return resumo


gold_linguagens = criar_gold_tecnologia(df_silver, "linguagens_uso", "Linguagem")
gold_bancos = criar_gold_tecnologia(df_silver, "bancos_dados_uso", "Banco de Dados")
gold_cloud = criar_gold_tecnologia(df_silver, "cloud_uso", "Cloud")
gold_bi = criar_gold_tecnologia(df_silver, "ferramentas_bi_uso", "BI")


gold_tecnologias = (
    gold_linguagens
    .unionByName(gold_bancos)
    .unionByName(gold_cloud)
    .unionByName(gold_bi)
    .cache()
)

gold_tecnologias.orderBy("ano", "categoria_tecnologia", F.desc("quantidade_profissionais")).show(40, truncate=False)

+----+--------------------+-----------------------------------------------------------------------------+------------------------+-----------------+
|ano |categoria_tecnologia|tecnologia                                                                   |quantidade_profissionais|percentual_adocao|
+----+--------------------+-----------------------------------------------------------------------------+------------------------+-----------------+
|2023|BI                  |Microsoft_PowerBI_#205                                                       |2154                    |58.22            |
|2023|BI                  |Looker_#211                                                                  |898                     |24.27            |
|2023|BI                  |Tableau_#207                                                                 |713                     |19.27            |
|2023|BI                  |Looker_Studio_Google_Data_Studio__#212                                       |6

## 9. Gold Inteligência Artificial

### Objetivo

Responder:

- Qual é o índice de adoção de Inteligência Artificial?
- Qual é o impacto percebido?
- IA é prioridade para as empresas?

Serão criados três indicadores:

- adoção de IA;
- prioridade empresarial;
- resultados/impacto percebido.

Quando `ia_resultados_empresa` não existir para determinado ano, não será criado um indicador artificial.

In [ ]:
ia_adocao = resumo_indicador(df_silver, "ia_adocao", "Adoção de IA")
ia_prioridade = resumo_indicador(df_silver, "ia_prioridade_empresa", "Prioridade de IA")
ia_resultados = resumo_indicador(df_silver, "ia_resultados_empresa", "Resultados / Impacto")

gold_ia = (
    ia_adocao
    .unionByName(ia_prioridade)
    .unionByName(ia_resultados)
    .cache()
)

gold_ia.orderBy("ano", "indicador", F.desc("quantidade")).show(40, truncate=False)

+----+--------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+----------+----------+
|ano |indicador           |categoria                                                                                                                                              |quantidade|percentual|
+----+--------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+----------+----------+
|2023|Adoção de IA        |Utiliza                                                                                                                                                |3772      |100.0     |
|2023|Prioridade de IA    |Mais ou menos... É uma das várias iniciativas que estamos impulsionando, mas não é uma prioridade (iniciativas isoladas e pouco foco).                 |275       |30

## 10. Gold Segmentações

### Objetivo

Responder:

- Existem diferenças relevantes entre regiões?
- Existem diferenças por senioridade?
- Existem diferenças por modelo de trabalho?

Para cada dimensão serão calculados:

- quantidade de profissionais;
- participação percentual;
- salário médio;
- salário mediano;
- percentual de adoção de IA.

In [ ]:
segmentacao_regiao = resumo_segmentacao(df_silver, "regiao", "Região")
segmentacao_senioridade = resumo_segmentacao(df_silver, "senioridade_grupo", "Senioridade")
segmentacao_modelo = resumo_segmentacao(df_silver, "modelo_trabalho_grupo", "Modelo de Trabalho")


gold_segmentacoes = (
    segmentacao_regiao
    .unionByName(
        segmentacao_senioridade
    )
    .unionByName(
        segmentacao_modelo
    )
    .cache()
)

gold_segmentacoes.orderBy("ano", "dimensao", F.desc("quantidade_profissionais")).show(40, truncate=False)

+----+------------------+---------------------+------------------------+----------+-------------+---------------+--------------------+
|ano |dimensao          |categoria            |quantidade_profissionais|percentual|salario_medio|salario_mediano|percentual_adocao_ia|
+----+------------------+---------------------+------------------------+----------+-------------+---------------+--------------------+
|2023|Modelo de Trabalho|Remoto               |2201                    |46.31     |11210.89     |10000.5        |100.0               |
|2023|Modelo de Trabalho|Híbrido              |1762                    |37.07     |10964.75     |10000.5        |100.0               |
|2023|Modelo de Trabalho|Presencial           |790                     |16.62     |7016.95      |5000.5         |100.0               |
|2023|Região            |Sudeste              |3173                    |61.39     |10905.37     |10000.5        |100.0               |
|2023|Região            |Sul                  |961     

## 11. Gold Desafios

### Objetivo

Responder:

- Quais oportunidades e desafios podem ser identificados para empresas que desejam investir em Dados e IA?

Serão considerados:

- desafios apontados por gestores;
- motivos para não utilizar IA.

As respostas multisseleção são separadas e transformadas em um ranking de frequência.

In [ ]:
def criar_gold_desafio(df, coluna, tipo_desafio):
    totais = (
        df
        .filter(
            F.col(coluna).isNotNull()
        )
        .groupBy("ano")
        .agg(
            F.count("*").alias(
                "respondentes_com_resposta"
            )
        )
    )

    explodido = explodir_multiselect(df, coluna, "desafio")

    return (
        explodido
        .groupBy(
            "ano",
            "desafio"
        )
        .agg(
            F.count("*").alias(
                "quantidade"
            )
        )
        .join(
            totais,
            on="ano",
            how="left"
        )
        .withColumn(
            "percentual",
            F.round(
                F.col("quantidade")
                / F.col(
                    "respondentes_com_resposta"
                )
                * 100,
                2
            )
        )
        .withColumn(
            "tipo_desafio",
            F.lit(
                tipo_desafio
            )
        )
        .select(
            "ano",
            "tipo_desafio",
            "desafio",
            "quantidade",
            "percentual",
        )
    )


desafios_gestao = criar_gold_desafio(df_silver, "desafios_gestor", "Gestão de Dados")
barreiras_ia = criar_gold_desafio(df_silver, "ia_motivos_nao_uso_empresa", "Adoção de IA")

gold_desafios = (desafios_gestao
    .unionByName(
        barreiras_ia
    )
    .cache()
)

gold_desafios.orderBy("ano", "tipo_desafio", F.desc("quantidade")).show(40, truncate=False)

+----+---------------+---------------------------------------------------------------------------------------+----------+----------+
|ano |tipo_desafio   |desafio                                                                                |quantidade|percentual|
+----+---------------+---------------------------------------------------------------------------------------+----------+----------+
|2023|Adoção de IA   |0                                                                                      |520       |63.18     |
|2023|Adoção de IA   |1                                                                                      |303       |36.82     |
|2023|Gestão de Dados|0                                                                                      |758       |86.73     |
|2023|Gestão de Dados|1                                                                                      |116       |13.27     |
|2024|Adoção de IA   |Falta de compreensão dos casos de uso.         

## 12. Controle de qualidade das tabelas Gold

O controle mantém o mesmo padrão simplificado da etapa Raw → Silver:

- quantidade de registros;
- quantidade por ano;
- visualização de uma amostra.

In [ ]:
GOLD_DATAFRAMES = {
    "tb_gold_mercado": gold_mercado,
    "tb_gold_diversidade": gold_diversidade,
    "tb_gold_tecnologias": gold_tecnologias,
    "tb_gold_ia": gold_ia,
    "tb_gold_segmentacoes": gold_segmentacoes,
    "tb_gold_desafios": gold_desafios,
}

for nome, df in GOLD_DATAFRAMES.items():

    print("=" * 80)
    print(nome)
    print("Registros:", df.count())

    df.groupBy("ano").count().orderBy("ano").show()

tb_gold_mercado
Registros: 116
+----+-----+
| ano|count|
+----+-----+
|2023|   39|
|2024|   38|
|2025|   39|
+----+-----+

tb_gold_diversidade
Registros: 122
+----+-----+
| ano|count|
+----+-----+
|2023|   40|
|2024|   38|
|2025|   44|
+----+-----+

tb_gold_tecnologias
Registros: 217
+----+-----+
| ano|count|
+----+-----+
|2023|   78|
|2024|   72|
|2025|   67|
+----+-----+

tb_gold_ia
Registros: 23
+----+-----+
| ano|count|
+----+-----+
|2023|    6|
|2024|    6|
|2025|   11|
+----+-----+

tb_gold_segmentacoes
Registros: 34
+----+-----+
| ano|count|
+----+-----+
|2023|   11|
|2024|   11|
|2025|   12|
+----+-----+

tb_gold_desafios
Registros: 120
+----+-----+
| ano|count|
+----+-----+
|2023|    4|
|2024|   69|
|2025|   47|
+----+-----+


## 13. Escrita das tabelas Gold no S3

Cada tabela será gravada em seu próprio diretório e particionada por `ano`.

Como os datasets Gold são pequenos, usamos `repartition(3, "ano")`
para reduzir a quantidade de arquivos Parquet.

In [ ]:
for nome, df in GOLD_DATAFRAMES.items():

    path = GOLD_CONFIG[nome]["path"]

    (
        df
        .repartition(
            3,
            "ano"
        )
        .write
        .mode("overwrite")
        .partitionBy("ano")
        .parquet(path)
    )

    print(nome, "gravada em", path)

tb_gold_mercado gravada em s3://tech-challenge-fase3-015006598133/gold/mercado/
tb_gold_diversidade gravada em s3://tech-challenge-fase3-015006598133/gold/diversidade/
tb_gold_tecnologias gravada em s3://tech-challenge-fase3-015006598133/gold/tecnologias/
tb_gold_ia gravada em s3://tech-challenge-fase3-015006598133/gold/ia/
tb_gold_segmentacoes gravada em s3://tech-challenge-fase3-015006598133/gold/segmentacoes/
tb_gold_desafios gravada em s3://tech-challenge-fase3-015006598133/gold/desafios/


## 14. Catalogação no Glue Data Catalog

As seis tabelas Gold serão registradas no banco `workspace`.

A função abaixo segue o mesmo padrão utilizado no notebook Raw → Silver.

In [ ]:
glue = boto3.client(
    "glue",
    region_name="us-east-1"
)


def tipo_glue(data_type):

    if isinstance(
        data_type,
        T.StringType
    ):
        return "string"

    if isinstance(
        data_type,
        T.IntegerType
    ):
        return "int"

    if isinstance(
        data_type,
        T.LongType
    ):
        return "bigint"

    if isinstance(
        data_type,
        T.DoubleType
    ):
        return "double"

    if isinstance(
        data_type,
        T.FloatType
    ):
        return "float"

    if isinstance(
        data_type,
        T.BooleanType
    ):
        return "boolean"

    if isinstance(
        data_type,
        T.TimestampType
    ):
        return "timestamp"

    return "string"


def catalogar_tabela_gold(nome_tabela, df, path):

    colunas_catalogo = [
        {
            "Name": campo.name,
            "Type": tipo_glue(
                campo.dataType
            )
        }
        for campo in df.schema.fields
        if campo.name != "ano"
    ]

    storage_descriptor = {
        "Columns": colunas_catalogo,
        "Location": path,
        "InputFormat": (
            "org.apache.hadoop.hive.ql.io.parquet."
            "MapredParquetInputFormat"
        ),
        "OutputFormat": (
            "org.apache.hadoop.hive.ql.io.parquet."
            "MapredParquetOutputFormat"
        ),
        "SerdeInfo": {
            "SerializationLibrary": (
                "org.apache.hadoop.hive.ql.io.parquet.serde."
                "ParquetHiveSerDe"
            )
        }
    }

    table_input = {
        "Name": nome_tabela,
        "TableType": "EXTERNAL_TABLE",
        "Parameters": {
            "classification": "parquet",
            "EXTERNAL": "TRUE"
        },
        "PartitionKeys": [
            {
                "Name": "ano",
                "Type": "int"
            }
        ],
        "StorageDescriptor": storage_descriptor
    }

    try:

        glue.get_table(
            DatabaseName=DATABASE,
            Name=nome_tabela
        )

        glue.update_table(
            DatabaseName=DATABASE,
            TableInput=table_input
        )

        print(
            nome_tabela,
            "atualizada no catálogo."
        )

    except glue.exceptions.EntityNotFoundException:

        glue.create_table(
            DatabaseName=DATABASE,
            TableInput=table_input
        )

        print(
            nome_tabela,
            "criada no catálogo."
        )


    # Registra/atualiza as partições
    for ano in [2023, 2024, 2025]:

        partition_input = {
            "Values": [
                str(ano)
            ],
            "StorageDescriptor": {
                **storage_descriptor,
                "Location": (
                    f"{path}ano={ano}/"
                )
            }
        }

        try:

            glue.create_partition(
                DatabaseName=DATABASE,
                TableName=nome_tabela,
                PartitionInput=partition_input
            )

        except glue.exceptions.AlreadyExistsException:

            glue.update_partition(
                DatabaseName=DATABASE,
                TableName=nome_tabela,
                PartitionValueList=[
                    str(ano)
                ],
                PartitionInput=partition_input
            )


for nome, df in GOLD_DATAFRAMES.items():

    catalogar_tabela_gold(
        nome_tabela=nome,
        df=df,
        path=GOLD_CONFIG[nome]["path"]
    )

tb_gold_mercado criada no catálogo.
tb_gold_diversidade criada no catálogo.
tb_gold_tecnologias criada no catálogo.
tb_gold_ia criada no catálogo.
tb_gold_segmentacoes criada no catálogo.
tb_gold_desafios criada no catálogo.


## 15. Validação final

As tabelas são lidas novamente pelo Glue Data Catalog para confirmar
que os arquivos e as partições estão acessíveis.

In [ ]:
for nome in GOLD_DATAFRAMES.keys():

    print("=" * 80)
    print(nome)

    df_validacao = (
        glueContext
        .create_data_frame_from_catalog(
            database=DATABASE,
            table_name=nome
        )
    )

    print("Registros:", df_validacao.count())

    df_validacao.groupBy("ano").count().orderBy("ano").show()

tb_gold_mercado
Registros: 116
+----+-----+
| ano|count|
+----+-----+
|2023|   39|
|2024|   38|
|2025|   39|
+----+-----+

tb_gold_diversidade
Registros: 122
+----+-----+
| ano|count|
+----+-----+
|2023|   40|
|2024|   38|
|2025|   44|
+----+-----+

tb_gold_tecnologias
Registros: 217
+----+-----+
| ano|count|
+----+-----+
|2023|   78|
|2024|   72|
|2025|   67|
+----+-----+

tb_gold_ia
Registros: 23
+----+-----+
| ano|count|
+----+-----+
|2023|    6|
|2024|    6|
|2025|   11|
+----+-----+

tb_gold_segmentacoes
Registros: 34
+----+-----+
| ano|count|
+----+-----+
|2023|   11|
|2024|   11|
|2025|   12|
+----+-----+

tb_gold_desafios
Registros: 120
+----+-----+
| ano|count|
+----+-----+
|2023|    4|
|2024|   69|
|2025|   47|
+----+-----+
